# Tutorial: Calculate geography and basic outcomes by LSOA

This notebook walks through the geography and basic outcomes step of the main calculations notebook.

It uses the base case scenario and shows more explicitly what calculations happen at each step. At the end, we show that the results are equivalent to the file that was saved in the main calculations notebook: `results_geo_outcomes_base.csv`.


__Aim: recreate the geography and basic outcomes__

In the main run, the geoprocessing, scenario and outcome model steps are combined in a single function, `calculate_geo_and_outcomes`. In this notebook we will break this function down into steps.

The code in this function is as follows:

In [1]:
# Add path to "classes" directory:
import sys
sys.path.append('../')

import inspect

from classes.add_net_benefit import calculate_geo_and_outcomes

In [2]:
print(inspect.getsource(calculate_geo_and_outcomes))

def calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
        dir_data='data'
    ):
    # Process geographic data (only needed when hospital data changes):
    geo = Geoprocessing(limit_to_england=True,
                        mt_travel_bias=mt_travel_bias,
                        new_mt_centres=new_mt_units,
                        dir_data=dir_data)
    geo.run()
    geodata = geo.combined_data.copy().reset_index()
    
    # Set up scenario
    scenario = Scenario({
        'name': 'temporary_name',
        'limit_to_england' : True
    })
    
    # Set up model
    model = Model(
        scenario=scenario,
        geodata=geodata
    )
    
    # Run model
    model.run()
    results = model.full_results
    return results



## Code setup

In [3]:
import os
import numpy as np
import pandas as pd

from classes.calc_scenario import load_scenario_parameters
from classes.geography_processing import Geoprocessing
from classes.model import Model
from classes.scenario import Scenario

In [4]:
dir_data = os.path.join('..', 'data')
dir_output = os.path.join('..', 'output')
file_scen_params = 'scenario_parameters_taunworc.csv'
file_geo_outcomes = 'results_geo_outcomes_'  # + scenario.csv

In [5]:
col_admissions = 'Admissions_2025_Est'

## Define scenario parameters

Load the scenario parameters from file:

In [6]:
p = os.path.join(dir_output, file_scen_params)
df_scen_full = load_scenario_parameters(p)

In [7]:
display(df_scen_full.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,mt_travel_bias_minutes
0,base,base,[],0.1
1,base_bias10min,base_bias10min,[],10.1
2,base_bias15min,base_bias15min,[],15.1
3,taunworc,taunworc,"[TA15DA, WR51DD]",0.1
4,taunworc_bias10min,taunworc_bias10min,"[TA15DA, WR51DD]",10.1
5,taunworc_bias15min,taunworc_bias15min,"[TA15DA, WR51DD]",15.1


Pick out the data for the geographic modelling scenario in this notebook:

In [8]:
# Pick out the data for this scenario geography name:
scen_geo_name = 'base'

# All rows with this geo name:
df = df_scen_full[df_scen_full['scenario_geoprocess_name'] == scen_geo_name]
# These parameters should be the same for all df in group:
mt_travel_bias = df.iloc[0]['mt_travel_bias_minutes']
new_mt_units = df.iloc[0]['new_mt_units']

# Display data:
display(df)
print('Scenario geography name:', scen_geo_name)
print('Travel bias for MT unit:', mt_travel_bias)
print('List of new MT units:', new_mt_units)

,scenario_name,scenario_geoprocess_name,new_mt_units,mt_travel_bias_minutes
0,base,base,[],0.1


Scenario geography name: base
Travel bias for MT unit: 0.1
List of new MT units: []


## Geography

The geography steps assign the nearest stroke unit in each case.

Unit selection steps for each area:
1. Find the minimum adjusted* travel time to any stroke unit, and find a list of the units with that adjusted time. If there is only one unit, select that unit. Otherwise, continue.
2. Out of the multiple nearest units by travel time, find which have the minimum travel distance. If there is only one unit, select that unit. Otherwise, continue.
4. Out of the multiple nearest units by time and distance, then arbitrarily select the first unit alphabetically by postcode.

\* adjusted travel time includes any travel bias, for example to favour MT units that are no more than X minutes farther away than a nearer IVT-only unit.

Initialise the geography object using the scenario parameters defined above:

In [9]:
# Process geographic data (only needed when hospital data changes):
geo = Geoprocessing(limit_to_england=True,
                    mt_travel_bias=mt_travel_bias,
                    new_mt_centres=new_mt_units,
                    dir_data=dir_data)

Note that here we have to provide the path to the directory containing the reference data. For the main notebook that creates all of the data, the data directory is the same as the default option. Here, the data directory is in the level above the one containing this notebook.

Keep a copy of the starting attributes:

In [10]:
attrs_start = dir(geo)

In the main notebook, we use the `run()` function. The code for this is as follows:

In [11]:
print(inspect.getsource(geo.run))

    def run(self):
        """
        Run all processing methods.
        """
        self.load_data()
        self.find_nearest_ivt_unit()
        self.find_nearest_mt_unit()
        self.find_nearest_transfer_mt_unit()
        self.collate_data()
        # self.save_processed_data()



Here we will run each of those steps individually and check the results at each stage.

### 1. load_data()

Load the data from file. This loads in data from the following sources in `dir_data`:

+ `hospitals`: 'stroke_hospitals_2025.csv'
+ `admissions`: 'admissions_2025_simple_scaled.csv'
+ `inter_hospital_time`: 'inter_hospital_pivoted_time.csv'
+ `lsoa_travel_time`: 'pivoted_time.csv'
+ `inter_hospital_distance`: 'inter_hospital_pivoted_distance.csv'
+ `lsoa_travel_distance`: 'pivoted_distance_km.csv'

All travel times are rounded to the nearest minute.

In [12]:
geo.load_data()

In [13]:
attrs_load = [a for a in dir(geo) if a not in attrs_start]

print(attrs_load)

['admissions', 'hospitals', 'inter_hospital_distance', 'inter_hospital_time', 'lsoa_travel_distance', 'lsoa_travel_time']


### 2. find_nearest_ivt_unit()

In [14]:
geo.find_nearest_ivt_unit()

In [15]:
attrs_nearest = [a for a in dir(geo) if a not in attrs_start + attrs_load]

print(attrs_nearest)

['IVT_hospitals', 'nearest_ivt_unit']


`IVT_hospitals` is a list of postcodes of units that offer IVT:

In [16]:
print(geo.IVT_hospitals[:5])

['B152TH', 'B662QT', 'BA13NG', 'BB23HH', 'BD96RJ']


`nearest_ivt_unit` is a DataFrame containing each LSOA's selected nearest IVT unit and its actual and adjusted travel times, flags from the selection process, and the "no bias" nearest IVT unit and travel time.

The "adjusted travel time" is used to select the unit and includes the travel bias to ensure that an MT unit is preferred. The "time" is the real travel time without the adjustment. The `no_bias` columns record the unit and travel time that would have been selected with no selection bias for MT units.

`unit_multiple_same_time` is set to 1 when an LSOA has multiple nearest IVT units by adjusted travel time. `unit_multiple_same_time_and_distance` is set to 1 when an LSOA still has multiple nearest IVT units by both adjusted travel time and by travel distance. When `unit_multiple_same_time_and_distance` is 1, then the selected unit in the `unit` column is the first option alphabetically in the list in `unit_multiple_same_time_and_distance_options`.

In [17]:
display(geo.nearest_ivt_unit.head(3))

,unit,adjusted_time,unit_multiple_same_time,unit_multiple_same_time_and_distance,unit_multiple_same_time_and_distance_options,time,unit_no_bias,time_no_bias
LSOA,,,,,,,,
ADUR001A,BN25BE,10.9,NaN,NaN,NaN,11,BN25BE,11
ADUR001B,BN25BE,11.9,NaN,NaN,NaN,12,BN112DH,12
ADUR001C,BN112DH,11.0,NaN,NaN,NaN,11,BN112DH,11


Check a few results where the flags are used:

In [18]:
cols_flag = ['unit_multiple_same_time', 'unit_multiple_same_time_and_distance']

for col in cols_flag:
    m = geo.nearest_ivt_unit[col] == 1
    print(col)
    print(len(geo.nearest_ivt_unit[m]))
    display(geo.nearest_ivt_unit[m].head(3))
    print('')

unit_multiple_same_time
590


,unit,adjusted_time,unit_multiple_same_time,unit_multiple_same_time_and_distance,unit_multiple_same_time_and_distance_options,time,unit_no_bias,time_no_bias
LSOA,,,,,,,,
AYLESBURYVALE015E,HP112TT,25.0,1.0,0.0,[],25,HP112TT,25
AYLESBURYVALE016B,HP112TT,26.0,1.0,0.0,[],26,HP112TT,26
AYLESBURYVALE016C,HP112TT,26.0,1.0,0.0,[],26,HP112TT,26



unit_multiple_same_time_and_distance
6


,unit,adjusted_time,unit_multiple_same_time,unit_multiple_same_time_and_distance,unit_multiple_same_time_and_distance_options,time,unit_no_bias,time_no_bias
LSOA,,,,,,,,
ISLINGTON012E,E11BB,10.9,1.0,1.0,"[E11BB, NW12BU]",11,E11BB,11
KINGSTONUPONTHAMES006D,SW170QT,14.9,1.0,1.0,"[SW170QT, SW66SX]",15,KT187EG,15
OLDHAM026F,BL97TD,19.0,1.0,1.0,"[BL97TD, SK27JE]",19,BL97TD,19


Only six LSOA have multiple options, so display all of those:

In [19]:
col_options = 'unit_multiple_same_time_and_distance_options'

m = geo.nearest_ivt_unit[col_options].str.len() > 0
print(col_options)
print(len(geo.nearest_ivt_unit[m]))
display(geo.nearest_ivt_unit[m])

unit_multiple_same_time_and_distance_options
6


,unit,adjusted_time,unit_multiple_same_time,unit_multiple_same_time_and_distance,unit_multiple_same_time_and_distance_options,time,unit_no_bias,time_no_bias
LSOA,,,,,,,,
ISLINGTON012E,E11BB,10.9,1.0,1.0,"[E11BB, NW12BU]",11,E11BB,11
KINGSTONUPONTHAMES006D,SW170QT,14.9,1.0,1.0,"[SW170QT, SW66SX]",15,KT187EG,15
OLDHAM026F,BL97TD,19.0,1.0,1.0,"[BL97TD, SK27JE]",19,BL97TD,19
SOUTHWARK009A,E11BB,6.9,1.0,1.0,"[E11BB, SE59RW]",7,E11BB,7
WANDSWORTH020D,SW170QT,7.9,1.0,1.0,"[SW170QT, SW66SX]",8,SW170QT,8
WESTMINSTER023C,NW12BU,9.9,1.0,1.0,"[NW12BU, SW66SX]",10,NW12BU,10


### 3. find_nearest_mt_unit()


The nearest MT unit and the transfer unit do not use a travel time bias as all units considered should offer MT. The selection does use the same logic of check minimum time, then minimum distance, then alphabetically.


In [20]:
geo.find_nearest_mt_unit()

In [21]:
attrs_mt = [a for a in dir(geo) if a not in
                 attrs_start + attrs_load + attrs_nearest]

print(attrs_mt)

['MT_hospitals', 'nearest_mt_unit']


These attributes are similar to those from the nearest_ivt_unit step.

In [22]:
print(geo.MT_hospitals[:5])

['B152TH', 'BN25BE', 'BS105NB', 'CB20QQ', 'CT13NG']


In [23]:
display(geo.nearest_mt_unit.head(3))

,time,unit,unit_multiple_same_time,unit_multiple_same_time_and_distance,unit_multiple_same_time_and_distance_options
LSOA,,,,,
ADUR001A,11,BN25BE,NaN,NaN,NaN
ADUR001B,12,BN25BE,NaN,NaN,NaN
ADUR001C,13,BN25BE,NaN,NaN,NaN


### 4. find_nearest_transfer_mt_unit()

In [24]:
geo.find_nearest_transfer_mt_unit()

In [25]:
attrs_transfer = [
    a for a in dir(geo) if a not in
    attrs_start + attrs_load + attrs_nearest + attrs_mt
]

print(attrs_transfer)

['transfer_mt_unit']


This time the DataFrame records the transfer unit selected for each IVT-only unit. It follows the same selection logic as before: smallest travel time, then smallest distance, then alphabetically.

In [26]:
display(geo.transfer_mt_unit.head(3))

,transfer_time,transfer_unit,transfer_unit_multiple_same_time,transfer_unit_multiple_same_time_and_distance,transfer_unit_multiple_same_time_and_distance_options,transfer_required
from_postcode,,,,,,
B152TH,0,B152TH,NaN,NaN,NaN,False
B662QT,7,B152TH,NaN,NaN,NaN,True
B714HJ,13,B152TH,NaN,NaN,NaN,True


Check the flags here:

In [27]:
cols_flag = ['transfer_unit_multiple_same_time',
             'transfer_unit_multiple_same_time_and_distance']

for col in cols_flag:
    m = geo.transfer_mt_unit[col] == 1
    print(col)
    print(len(geo.transfer_mt_unit[m]))
    display(geo.transfer_mt_unit[m].head(3))
    print('')

transfer_unit_multiple_same_time
2


,transfer_time,transfer_unit,transfer_unit_multiple_same_time,transfer_unit_multiple_same_time_and_distance,transfer_unit_multiple_same_time_and_distance_options,transfer_required
from_postcode,,,,,,
KT160PZ,32,SW170QT,1.0,0.0,[],True
SN36BB,41,OX39DU,1.0,0.0,[],True



transfer_unit_multiple_same_time_and_distance
0


,transfer_time,transfer_unit,transfer_unit_multiple_same_time,transfer_unit_multiple_same_time_and_distance,transfer_unit_multiple_same_time_and_distance_options,transfer_required
from_postcode,,,,,,


Only two IVT-only units have multiple transfer units at the same travel time away, and all of those ties are broken by the distance check.

### 5. collate_data()

This step gathers the results from the previous geography steps into one place. For each LSOA, it gathers the selected nearest IVT unit and travel time, its transfer unit and travel time, and the mothership MT unit and travel time. It also gathers the selection flags and background info on the LSOAs: name, country, and number of stroke admissions.

In [28]:
geo.collate_data()

In [29]:
attrs_collate = [
    a for a in dir(geo) if a not in 
    attrs_start + attrs_load + attrs_nearest + attrs_mt + attrs_transfer
]

print(attrs_collate)

['combined_data']


In [30]:
display(geo.combined_data.head(3))

,nearest_ivt_unit,nearest_ivt_adjusted_time,nearest_ivt_unit_multiple_same_time,nearest_ivt_unit_multiple_same_time_and_distance,nearest_ivt_unit_multiple_same_time_and_distance_options,nearest_ivt_time,nearest_ivt_unit_no_bias,nearest_ivt_time_no_bias,nearest_mt_time,nearest_mt_unit,...,transfer_time,transfer_unit,transfer_unit_multiple_same_time,transfer_unit_multiple_same_time_and_distance,transfer_unit_multiple_same_time_and_distance_options,transfer_required,area_lower,Admissions,England,Admissions_2025_Est
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,BN25BE,10.9,NaN,NaN,NaN,11,BN25BE,11,11,BN25BE,...,0,BN25BE,NaN,NaN,NaN,False,Adur 001A,1.333333,1,1.49403
ADUR001B,BN25BE,11.9,NaN,NaN,NaN,12,BN112DH,12,12,BN25BE,...,0,BN25BE,NaN,NaN,NaN,False,Adur 001B,1.666667,1,1.86762
ADUR001C,BN112DH,11.0,NaN,NaN,NaN,11,BN112DH,11,13,BN25BE,...,24,BN25BE,NaN,NaN,NaN,True,Adur 001C,2.000000,1,2.24110


Check the columns present in the results:

In [31]:
print(geo.combined_data.columns)

Index(['nearest_ivt_unit', 'nearest_ivt_adjusted_time',
       'nearest_ivt_unit_multiple_same_time',
       'nearest_ivt_unit_multiple_same_time_and_distance',
       'nearest_ivt_unit_multiple_same_time_and_distance_options',
       'nearest_ivt_time', 'nearest_ivt_unit_no_bias',
       'nearest_ivt_time_no_bias', 'nearest_mt_time', 'nearest_mt_unit',
       'nearest_mt_unit_multiple_same_time',
       'nearest_mt_unit_multiple_same_time_and_distance',
       'nearest_mt_unit_multiple_same_time_and_distance_options',
       'transfer_time', 'transfer_unit', 'transfer_unit_multiple_same_time',
       'transfer_unit_multiple_same_time_and_distance',
       'transfer_unit_multiple_same_time_and_distance_options',
       'transfer_required', 'area_lower', 'Admissions', 'England',
       'Admissions_2025_Est'],
      dtype='str')


Check the LSOA background info:

In [34]:
cols_background = ['area_lower', 'Admissions', 'England',
                   'Admissions_2025_Est']

display(geo.combined_data[cols_background].head(3))

,area_lower,Admissions,England,Admissions_2025_Est
LSOA,,,,
ADUR001A,Adur 001A,1.333333,1,1.49403
ADUR001B,Adur 001B,1.666667,1,1.86762
ADUR001C,Adur 001C,2.000000,1,2.24110


There are two admissions columns. `Admissions` is based on the SSNAP data. `Admissions_2025_Est` is projections based on a fit to the numbers of people in each age band and deprivation quantile.

The column `England` should contain only 1 because we specified `limit_to_england`.

## Scenario

Next, we use the `scenario` class to store fixed parameters for the model. We use a throwaway name because it doesn't get used anywhere following the object initialisation.

In [35]:
# Set up scenario
scenario = Scenario({
    'name': 'temporary_name',
    'limit_to_england' : True
})

Pick out the attributes set up here and display them as a Series:

In [36]:
attrs_scen = [a for a in dir(scenario) if not a.startswith('__')]

s_scen = pd.Series()
for a in attrs_scen:
    s_scen[a] = getattr(scenario, a)

display(s_scen)

limit_to_england                                 True
name                                   temporary_name
process_MT_net_delay                               60
process_ambulance_on_scene_duration                30
process_time_ambulance_response                    20
process_time_arrival_to_needle                     30
process_time_arrival_to_puncture                   90
process_time_call_ambulance                        30
save_lsoa_results                               False
dtype: object

Most of these fixed parameters are process times. These are fixed pathway times that are added to the travel times to calculate the times to treatment in the outcome model.

## Outcome model

Now we can combine the process times from the Scenario with the travel times from Geoprocessing.

Make sure that the geography results are in the right format:

In [37]:
geodata = geo.combined_data.copy().reset_index()

Set up the outcome model object:

In [38]:
# Set up model
model = Model(
    scenario=scenario,
    geodata=geodata
)

In the main notebook, we use the `run()` function which contains the following code:

In [39]:
print(inspect.getsource(model.run))

    def run(self):
        """
        Run the model.
        """

        # set up table and add results
        self.full_results = self.geodata.copy(deep=True)

        self.add_drip_ship()
        self.add_mothership()

        # Reindex on LSOA
        self.full_results.set_index('LSOA', inplace=True)
        self.summary_results = self.full_results.describe().T


        self.save_results()



We'll run through each step separately and keep track of what's created.

Keep a copy of the starting attributes:

In [40]:
attrs_m_start = dir(model)

### 1. full_results

Copy in the geography results so that we can add to the same DataFrame:

In [41]:
model.full_results = model.geodata.copy(deep=True)

In [42]:
attrs_m_geo = [a for a in dir(model) if a not in attrs_m_start]
cols_geo = model.full_results.columns.tolist()

print(attrs_m_geo)

['full_results']


### 2. add_drip_ship()

In [43]:
model.add_drip_ship()

attrs_m_ds = [a for a in dir(model) if a not in attrs_m_start + attrs_m_geo]
cols_ds = [c for c in model.full_results.columns.tolist() if c not in cols_geo]

The input mRS distributions might be improperly labelled.


In [44]:
print(attrs_m_ds)

[]


There are no new attributes! This is because the new results have been placed straight into `full_results`. View the new column names:

In [45]:
print(cols_ds)

['drip_ship_ivt_time', 'drip_ship_mt_time', 'drip_ship_mt_travel_time', 'drip_ship_mt_travel_time_with_delay', 'nlvo_no_treatment_mrs_0-2', 'nlvo_no_treatment_utility', 'lvo_no_treatment_mrs_0-2', 'lvo_no_treatment_utility', 'nlvo_drip_ship_ivt_mrs_0-2', 'nlvo_drip_ship_ivt_mrs_shift', 'nlvo_drip_ship_ivt_utility', 'nlvo_drip_ship_ivt_utility_shift', 'lvo_drip_ship_ivt_mrs_0-2', 'lvo_drip_ship_ivt_mrs_shift', 'lvo_drip_ship_ivt_utility', 'lvo_drip_ship_ivt_utility_shift', 'lvo_drip_ship_mt_mrs_0-2', 'lvo_drip_ship_mt_mrs_shift', 'lvo_drip_ship_mt_utility', 'lvo_drip_ship_mt_utility_shift', 'lvo_drip_ship_ivt_mt_mrs_0-2', 'lvo_drip_ship_ivt_mt_mrs_shift', 'lvo_drip_ship_ivt_mt_utility', 'lvo_drip_ship_ivt_mt_utility_shift']


In [46]:
display(model.full_results[cols_ds].head(3))

,drip_ship_ivt_time,drip_ship_mt_time,drip_ship_mt_travel_time,drip_ship_mt_travel_time_with_delay,nlvo_no_treatment_mrs_0-2,nlvo_no_treatment_utility,lvo_no_treatment_mrs_0-2,lvo_no_treatment_utility,nlvo_drip_ship_ivt_mrs_0-2,nlvo_drip_ship_ivt_mrs_shift,...,lvo_drip_ship_ivt_utility,lvo_drip_ship_ivt_utility_shift,lvo_drip_ship_mt_mrs_0-2,lvo_drip_ship_mt_mrs_shift,lvo_drip_ship_mt_utility,lvo_drip_ship_mt_utility_shift,lvo_drip_ship_ivt_mt_mrs_0-2,lvo_drip_ship_ivt_mt_mrs_shift,lvo_drip_ship_ivt_mt_utility,lvo_drip_ship_ivt_mt_utility_shift
0,121,181,11,11,0.58,0.59964,0.265,0.33261,0.700497,-0.611717,...,0.387616,0.055006,0.429398,-0.785154,0.488194,0.155584,0.429398,-0.785154,0.488194,0.155584
1,122,182,12,12,0.58,0.59964,0.265,0.33261,0.700046,-0.609338,...,0.387340,0.054730,0.428752,-0.781922,0.487596,0.154986,0.428752,-0.781922,0.487596,0.154986
2,121,265,35,95,0.58,0.59964,0.265,0.33261,0.700497,-0.611717,...,0.387616,0.055006,0.376218,-0.520433,0.438498,0.105888,0.376218,-0.520433,0.438498,0.105888


For each LSOA, we have calculated the treatment times to IVT and MT in the drip-and-ship scenario (`drip_ship_ivt_time`, `drip_ship_mt_time`). We have also calculated the travel time including any transfer and delays at the first unit (`drip_ship_mt_travel_time`, `drip_ship_mt_travel_time_with_delay`).

The rest of the columns are the results of the outcome model with those treatment times.

Outcomes:
+ proportion of patients with mRS score of 0-2 inclusive,
+ the change in mRS score,
+ the utility, and
+ the change in utility.

Treatments:
+ IVT only
+ MT only
+ both IVT and MT

Patient types:
+ nLVO
+ LVO

At this stage there are no assumptions about the full cohort, for example the proportion of patients who have an LVO or who receive treatment.

### 2. add_mothership()

This is a repeat of the drip-and-ship method but using the treatments times in the mothership case (everybody goes directly to their nearest MT unit).

In [47]:
model.add_mothership()

attrs_m_ms = [a for a in dir(model) if a not in
              attrs_m_start + attrs_m_geo + attrs_m_ds]
cols_ms = [c for c in model.full_results.columns if c not in
           cols_geo + cols_ds]

The input mRS distributions might be improperly labelled.


In [48]:
print(attrs_m_ms)

[]


In [49]:
print(cols_ms)

['mothership_ivt_time', 'mothership_mt_time', 'nlvo_mothership_ivt_mrs_0-2', 'nlvo_mothership_ivt_mrs_shift', 'nlvo_mothership_ivt_utility', 'nlvo_mothership_ivt_utility_shift', 'lvo_mothership_ivt_mrs_0-2', 'lvo_mothership_ivt_mrs_shift', 'lvo_mothership_ivt_utility', 'lvo_mothership_ivt_utility_shift', 'lvo_mothership_mt_mrs_0-2', 'lvo_mothership_mt_mrs_shift', 'lvo_mothership_mt_utility', 'lvo_mothership_mt_utility_shift', 'lvo_mothership_ivt_mt_mrs_0-2', 'lvo_mothership_ivt_mt_mrs_shift', 'lvo_mothership_ivt_mt_utility', 'lvo_mothership_ivt_mt_utility_shift']


In [50]:
display(model.full_results[cols_ms].head(3))

,mothership_ivt_time,mothership_mt_time,nlvo_mothership_ivt_mrs_0-2,nlvo_mothership_ivt_mrs_shift,nlvo_mothership_ivt_utility,nlvo_mothership_ivt_utility_shift,lvo_mothership_ivt_mrs_0-2,lvo_mothership_ivt_mrs_shift,lvo_mothership_ivt_utility,lvo_mothership_ivt_utility_shift,lvo_mothership_mt_mrs_0-2,lvo_mothership_mt_mrs_shift,lvo_mothership_mt_utility,lvo_mothership_mt_utility_shift,lvo_mothership_ivt_mt_mrs_0-2,lvo_mothership_ivt_mt_mrs_shift,lvo_mothership_ivt_mt_utility,lvo_mothership_ivt_mt_utility_shift
0,121,181,0.700497,-0.611717,0.711035,0.111395,0.324881,-0.273106,0.387616,0.055006,0.429398,-0.785154,0.488194,0.155584,0.429398,-0.785154,0.488194,0.155584
1,122,182,0.700046,-0.609338,0.710625,0.110985,0.324590,-0.271595,0.387340,0.054730,0.428752,-0.781922,0.487596,0.154986,0.428752,-0.781922,0.487596,0.154986
2,123,183,0.699596,-0.606957,0.710215,0.110575,0.324300,-0.270086,0.387064,0.054454,0.428107,-0.778692,0.486998,0.154388,0.428107,-0.778692,0.486998,0.154388


### 3. Reindex

Finally, get the DataFrame ready:

In [51]:
# Reindex on LSOA
model.full_results.set_index('LSOA', inplace=True)

This DataFrame is the main results from this process.

### 4. [extra] summary results

The method also calculates some summary statistics.

In [52]:
model.summary_results = model.full_results.describe().T

In [53]:
display(model.summary_results.head(3))

,count,mean,std,min,25%,50%,75%,max
nearest_ivt_adjusted_time,32843.0,13.102600,7.955004,-0.1,7.0,11.9,17.0,67.9
nearest_ivt_unit_multiple_same_time,590.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
nearest_ivt_unit_multiple_same_time_and_distance,590.0,0.010169,0.100415,0.0,0.0,0.0,0.0,1.0


Finally, the method runs `save_results()`. However this doesn't do anything because in `scenario` we set the option to not save the results in the `model` object.

## Results

Take the results out of the model object and round the values:

In [54]:
results = model.full_results

# Round outcome results:
cols_mrs = [c for c in results.columns if (
            ('mrs' in c) | ('improvement' in c))]
cols_util = [c for c in results.columns if ('util' in c)]
results[cols_mrs] = results[cols_mrs].round(3)
results[cols_util] = results[cols_util].round(5)

This `results` is the same as the output from the `calculate_geo_and_outcomes` function in the main notebook.

View results:

In [55]:
display(results.head(3))

,nearest_ivt_unit,nearest_ivt_adjusted_time,nearest_ivt_unit_multiple_same_time,nearest_ivt_unit_multiple_same_time_and_distance,nearest_ivt_unit_multiple_same_time_and_distance_options,nearest_ivt_time,nearest_ivt_unit_no_bias,nearest_ivt_time_no_bias,nearest_mt_time,nearest_mt_unit,...,lvo_mothership_ivt_utility,lvo_mothership_ivt_utility_shift,lvo_mothership_mt_mrs_0-2,lvo_mothership_mt_mrs_shift,lvo_mothership_mt_utility,lvo_mothership_mt_utility_shift,lvo_mothership_ivt_mt_mrs_0-2,lvo_mothership_ivt_mt_mrs_shift,lvo_mothership_ivt_mt_utility,lvo_mothership_ivt_mt_utility_shift
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,BN25BE,10.9,NaN,NaN,NaN,11,BN25BE,11,11,BN25BE,...,0.38762,0.05501,0.429,-0.785,0.48819,0.15558,0.429,-0.785,0.48819,0.15558
ADUR001B,BN25BE,11.9,NaN,NaN,NaN,12,BN112DH,12,12,BN25BE,...,0.38734,0.05473,0.429,-0.782,0.48760,0.15499,0.429,-0.782,0.48760,0.15499
ADUR001C,BN112DH,11.0,NaN,NaN,NaN,11,BN112DH,11,13,BN25BE,...,0.38706,0.05445,0.428,-0.779,0.48700,0.15439,0.428,-0.779,0.48700,0.15439


The results contain these columns:

In [56]:
print(results.columns)

Index(['nearest_ivt_unit', 'nearest_ivt_adjusted_time',
       'nearest_ivt_unit_multiple_same_time',
       'nearest_ivt_unit_multiple_same_time_and_distance',
       'nearest_ivt_unit_multiple_same_time_and_distance_options',
       'nearest_ivt_time', 'nearest_ivt_unit_no_bias',
       'nearest_ivt_time_no_bias', 'nearest_mt_time', 'nearest_mt_unit',
       'nearest_mt_unit_multiple_same_time',
       'nearest_mt_unit_multiple_same_time_and_distance',
       'nearest_mt_unit_multiple_same_time_and_distance_options',
       'transfer_time', 'transfer_unit', 'transfer_unit_multiple_same_time',
       'transfer_unit_multiple_same_time_and_distance',
       'transfer_unit_multiple_same_time_and_distance_options',
       'transfer_required', 'area_lower', 'Admissions', 'England',
       'Admissions_2025_Est', 'drip_ship_ivt_time', 'drip_ship_mt_time',
       'drip_ship_mt_travel_time', 'drip_ship_mt_travel_time_with_delay',
       'nlvo_no_treatment_mrs_0-2', 'nlvo_no_treatment_utility',

## Check results

Compare with results on file:

In [57]:
# Use converters to turn units column from strings to lists.
# Assumes that no postcodes contain spaces!
cols_options = ['nearest_ivt_unit_multiple_same_time_and_distance_options',
                'nearest_mt_unit_multiple_same_time_and_distance_options',
                'transfer_unit_multiple_same_time_and_distance_options']
converters = dict()
for col in cols_options:
    converters[col] = lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()
        if len(x) > 0 else np.nan
    )

f = f'{file_geo_outcomes}{scen_geo_name}.csv'
p = os.path.join(dir_output, f)
results_loaded = pd.read_csv(p, index_col=0, converters=converters)

/tmp/ipykernel_31380/1379276172.py:15: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  results_loaded = pd.read_csv(p, index_col=0, converters=converters)


If the results calculated here are identical to the ones on file, then the following cell will print True:

In [58]:
results.equals(results_loaded)

True

## Conclusion

We have shown the step-by-step process for assigning the stroke units to each LSOA and calculating their outcomes based on the treatment times including fixed pathway times and geography-dependent travel times.